In [1]:
import torch
!pip uninstall -y sympy
!pip install sympy==1.13.3
from torch import nn
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms, models
import torch.optim as optim
import requests
import random
import kagglehub
import datetime
import os
import json
import numpy as np
from datetime import datetime
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from timeit import default_timer as timer


Found existing installation: sympy 1.13.3
Uninstalling sympy-1.13.3:
  Successfully uninstalled sympy-1.13.3
  Using cached sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.13.3-py3-none-any.whl (6.2 MB)


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 134
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
device

'cuda'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
image_path = Path(kagglehub.dataset_download("mengcius/cinic10"))
train_dir = image_path / "train"
valid_dir = image_path / "valid"
test_dir = image_path / "test"

Using Colab cache for faster access to the 'cinic10' dataset.


In [6]:
data_transform = transforms.Compose([
    transforms.ToTensor()
])

train_data = datasets.ImageFolder(root=train_dir,
                                  transform=data_transform,
                                  target_transform=None)

valid_data = test_data = datasets.ImageFolder(root=valid_dir,
                                 transform=data_transform)

# test_data = datasets.ImageFolder(root=test_dir,
#                                  transform=data_transform)

In [7]:
def build_resnet18(num_classes=10, weights="IMAGENET1K_V1", dropout=0):
  model = models.resnet18(weights=weights)
  # zmiany ktore pozwalją lepiej dzialac na obrazkach 32x32, bo defaultowo sa 224x224
  model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
  model.maxpool = nn.Identity()

  # Add dropout
  model.fc = nn.Sequential(
    nn.Dropout(dropout),
    nn.Linear(model.fc.in_features, num_classes)
)
  return model

In [ ]:
def build_densenet121(num_classes=10, weights="IMAGENET1K_V1", dropout=0):
    model = models.densenet121(weights=weights)

    # Adjust for 32x32 images
    model.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.features.pool0 = nn.Identity()

    # Add dropout
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(num_ftrs, num_classes)
    )

    return model

In [8]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module):
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            test_pred_logits = model(X)
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))

    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [9]:
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          save_dir: str,
          scheduler=None):

    results = {"train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    for epoch in tqdm(range(epochs)):
        best_acc_test = 0
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer)
        test_loss, test_acc = test_step(model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn)

        # if scheduler is not None:
        #     scheduler.step()
        #     current_lr = optimizer.param_groups[0]['lr']
        # else:
        #     current_lr = "N/A"

        if test_acc > best_acc_test:
            best_acc_test = test_acc
            best_model = model

        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"valid_loss: {test_loss:.4f} | "
            f"valid_acc: {test_acc:.4f}"
        )
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

    torch.save(best_model.state_dict(), save_dir)
    return results

In [11]:
# take 75% of train/valid data
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

class_indices_train = defaultdict(list)
class_indices_valid = defaultdict(list)

for i, (_, label) in enumerate(train_data):
    class_indices_train[label].append(i)

for i, (_, label) in enumerate(valid_data):
    class_indices_valid[label].append(i)

selected_train = []
selected_valid = []
for label, inds in class_indices_train.items():
    random.shuffle(inds)
    selected_train.extend(inds[:int(0.75 * len(inds))])

for label, inds in class_indices_valid.items():
    random.shuffle(inds)
    selected_valid.extend(inds[:int(0.75 * len(inds))])

subset_train = Subset(train_data, selected_train)
subset_valid = Subset(valid_data, selected_valid)

train_dataloader = DataLoader(
    subset_train,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True
)

valid_dataloader = DataLoader(
    subset_valid,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True
)

In [13]:
# uzupełnić przed puszczeniem
MODEL = "resnet18"
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()
NUM_EPOCHS = 40
LEARNING_RATE = 0.1
WEIGHT_DECAY = 1e-4
DROPOUT = 0
PRETRAINED_WEIGHTS = "IMAGENET1K_V1"
DATA_AUG = "None"

model = build_resnet18(10, weights = PRETRAINED_WEIGHTS, dropout=DROPOUT).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    momentum = 0.9,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

timestamp = int(datetime.now().timestamp())
result_path_model = f"drive/MyDrive/data/{MODEL}_{timestamp}"
result_path = f"drive/MyDrive/data/res_{MODEL}_{timestamp}.json"

start_time = timer()
model_results = train(model=model,
                        train_dataloader=train_dataloader,
                        test_dataloader=valid_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        save_dir=result_path_model,
                        scheduler=None)
end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

result_data = {
    "model": MODEL,
    "results": model_results,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "num_workers": NUM_WORKERS,
    "training_time": end_time-start_time,
    "learning_rate": LEARNING_RATE,
    "dropout": DROPOUT,
    "weight_decay": WEIGHT_DECAY,
    "pretrained_weights": PRETRAINED_WEIGHTS,
    "optimizer": "SGD",
    "data_augmentation": DATA_AUG}

with open(result_path, 'w') as fp:
    json.dump(result_data, fp)

  0%|          | 0/40 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 2.2135 | train_acc: 0.2492 | valid_loss: 1.8043 | valid_acc: 0.3431
Epoch: 2 | train_loss: 1.7645 | train_acc: 0.3512 | valid_loss: 1.7014 | valid_acc: 0.3699
Epoch: 3 | train_loss: 1.6538 | train_acc: 0.3862 | valid_loss: 1.7232 | valid_acc: 0.3727
Epoch: 4 | train_loss: 1.5575 | train_acc: 0.4243 | valid_loss: 1.5635 | valid_acc: 0.4224
Epoch: 5 | train_loss: 1.4491 | train_acc: 0.4659 | valid_loss: 1.4725 | valid_acc: 0.4641
Epoch: 6 | train_loss: 1.3445 | train_acc: 0.5093 | valid_loss: 1.3242 | valid_acc: 0.5157
Epoch: 7 | train_loss: 1.2435 | train_acc: 0.5479 | valid_loss: 1.3495 | valid_acc: 0.5145
Epoch: 8 | train_loss: 1.1612 | train_acc: 0.5805 | valid_loss: 1.4045 | valid_acc: 0.4913
Epoch: 9 | train_loss: 1.0839 | train_acc: 0.6084 | valid_loss: 1.2732 | valid_acc: 0.5491
Epoch: 10 | train_loss: 1.0243 | train_acc: 0.6305 | valid_loss: 1.1451 | valid_acc: 0.5904
Epoch: 11 | train_loss: 0.9632 | train_acc: 0.6560 | valid_loss: 1.1359 | valid_acc: 0.59